In [1]:
import os
import requests
import pandas as pd

In [2]:
api="https://dadosabertos.camara.leg.br/api/v2"
arquivos= "https://dadosabertos.camara.leg.br/arquivos"

os.makedirs("dados", exist_ok=True)

In [3]:
def pegar(url):
    return requests.get(url, headers={"Accept": "application/json"}).json()["dados"]


In [ ]:
#votações
votacoes = pd.read_csv(arquivos + "/votacoes/csv/votacoes-2025.csv", sep=";", dtype=str)
vot_prop = pd.read_csv(arquivos + "/votacoesProposicoes/csv/votacoesProposicoes-2025.csv", sep=";", dtype=str)
votos = pd.read_csv(arquivos + "/votacoesVotos/csv/votacoesVotos-2025.csv", sep=";", dtype=str)

votacoes = votacoes[votacoes["siglaOrgao"] == "PLEN"]
vot_prop = vot_prop[vot_prop["idVotacao"].isin(votacoes["id"])]
vot_prop = vot_prop[vot_prop["proposicao_siglaTipo"].isin(["PL", "PLP", "PEC"])]
votos = votos[votos["idVotacao"].isin(vot_prop["idVotacao"])]

# votacao sem votos fica de fora
votacoes = votacoes[votacoes["id"].isin(votos["idVotacao"])]
vot_prop = vot_prop[vot_prop["idVotacao"].isin(votos["idVotacao"])]

proposicoes = vot_prop.drop_duplicates("proposicao_id")

# o partido é o do último voto do deputado em 2025
deputados = votos.drop_duplicates("deputado_id", keep="last")


In [ ]:
#autores
autores = []
for id_prop in proposicoes["proposicao_id"]:
    for autor in pegar(f"{api}/proposicoes/{id_prop}/autores"):
        if "/deputados/" in autor["uri"]:  # só autores que são deputados
            id_dep = autor["uri"].split("/")[-1]
            autores.append([id_prop, id_dep, autor["ordemAssinatura"]])


In [ ]:
#frentes
frentes = []
pagina = 1
while True:
    lista = pegar(f"{api}/frentes?idLegislatura=57&itens=100&pagina={pagina}")
    if not lista:
        break
    frentes += lista
    pagina += 1

membros = []
for frente in frentes:
    for m in pegar(f"{api}/frentes/{frente['id']}/membros"):
        membros.append([frente["id"], frente["titulo"], m["id"], m["titulo"]])



In [ ]:
# salvar os csv
deputados.to_csv("dados/deputados.csv", index=False,
                 columns=["deputado_id", "deputado_nome", "deputado_siglaPartido", "deputado_siglaUf"])
proposicoes.to_csv("dados/proposicoes.csv", index=False,
                   columns=["proposicao_id", "proposicao_siglaTipo", "proposicao_numero",
                            "proposicao_ano", "proposicao_ementa"])
votacoes.to_csv("dados/votacoes.csv", index=False,
                columns=["id", "data", "descricao", "aprovacao", "votosSim", "votosNao"])
vot_prop.to_csv("dados/votacao_proposicao.csv", index=False, columns=["idVotacao", "proposicao_id"])
votos.to_csv("dados/votos.csv", index=False, columns=["idVotacao", "deputado_id", "voto"])

pd.DataFrame(autores, columns=["idProposicao", "idDeputado", "ordem"]) \
    .to_csv("dados/autores.csv", index=False)
pd.DataFrame(membros, columns=["idFrente", "titulo", "idDeputado", "cargo"]) \
    .to_csv("dados/membros_frente.csv", index=False)


In [ ]:
for arquivo in sorted(os.listdir("dados")):
    print(arquivo, len(pd.read_csv("dados/" + arquivo)))
